# S1 - Independent-family scaling is the lever (not capacity)
**Question.** In-distribution feature/capacity levers were NULL. Does zero-shot transfer instead rise with the number of INDEPENDENT RBP families seen? (The interpolation-in-RBP-space reframe: ~4 wells cannot interpolate; ~99-1744 mmseqs families can.)

*Data: `mmpartnet_out/binding_x_p{1,2,3}.json` (committed); family curve computed on a CUDA GPU, results committed under `mmpartnet_out/famscale/` (see provenance in the last cell).*

## Definitions
- **P1 dim-sweep** (widen per-residue token dim), **P2 STRING-fusion** (+PPI PE), **P3 capacity** (depth) -- all measured vs the RNA-only baseline AUPRC (gap_vs_rna_only).
- **family curve**: train on N independent mmseqs families, eval HELD-OUT families; sweep N.

In [1]:
import json; from pathlib import Path; from IPython.display import Markdown, display
OUT = Path('..') / '..' / 'mmpartnet_out'
def J(n): return json.loads((OUT/n).read_text(encoding='utf-8'))
tbl = '| lever | method | gap vs RNA-only AUPRC |\n|---|---|---|\n'
for f, lev in [('binding_x_p1_dimsweep.json','P1 widen tokens'),
               ('binding_x_p2_stringfusion.json','P2 +STRING'),
               ('binding_x_p3_capacity.json','P3 depth')]:
    d = J(f)
    for mname, mv in d['methods'].items():
        g = mv.get('gap_vs_rna_only')
        if g is not None: tbl += f'| {lev} | {mname} | {g:+.3f} |\n'
display(Markdown(tbl))
display(Markdown('All levers hover near 0 (in-distribution) -> capacity/features are NOT the lever.'))

| lever | method | gap vs RNA-only AUPRC |
|---|---|---|
| P1 widen tokens | perres32_ref | +0.016 |
| P1 widen tokens | perres_full_32 | +0.017 |
| P1 widen tokens | perres_full_128 | +0.017 |
| P1 widen tokens | perres_full_256 | +0.017 |
| P1 widen tokens | perres_full_640 | +0.015 |
| P2 +STRING | perres32 | +0.017 |
| P2 +STRING | perres32+string | +0.015 |
| P2 +STRING | perres32+prott5 | +0.016 |
| P2 +STRING | perres32+both | +0.015 |
| P3 depth | perres_L2 | +0.017 |
| P3 depth | perres_L4 | +0.016 |
| P3 depth | bidir_L2 | +0.017 |
| P3 depth | bidir_L4 | +0.017 |


All levers hover near 0 (in-distribution) -> capacity/features are NOT the lever.

In [2]:
# family-scaling curve from the committed 5090 results (mmpartnet_out/famscale/famscale_N/0/)
import csv, os
from pathlib import Path as _P
FAM = _P(os.environ.get('ML4RG_FAMSCALE', OUT / 'famscale'))
def _read(N):
    f = FAM / f'famscale_{N}' / '0' / 'val_metrics.csv'
    if not f.exists(): return None
    rows = [r for r in csv.DictReader(open(f)) if r.get('MCC') not in (None,'')]
    if not rows: return None
    acc = [float(r['Accuracy']) for r in rows]; mcc = [float(r['MCC']) for r in rows]
    return {'nEp': len(rows), 'bestAcc': max(acc), 'bestMCC': max(mcc), 'lastMCC': mcc[-1]}
tbl = '| N families | epochs | best Acc | best MCC | last-epoch MCC |\n|---|---|---|---|---|\n'
for N in (10,25,50,100,200):
    r = _read(N)
    tbl += (f'| {N} | {r["nEp"]} | {r["bestAcc"]:.3f} | {r["bestMCC"]:.3f} | {r["lastMCC"]:+.3f} |\n'
            if r else f'| {N} | - | (pending) | | |\n')
display(Markdown(tbl))
display(Markdown('_Metric = MCC / Accuracy on the FIXED held-out families. F1 is omitted: it is degenerate here (peaks at epoch 0, i.e. before learning). Near-chance across N (Acc ~0.51-0.54, MCC ~0.02-0.10), NO clean monotonic rise; last-epoch MCC ~0 or negative._'))

| N families | epochs | best Acc | best MCC | last-epoch MCC |
|---|---|---|---|---|
| 10 | 4 | 0.531 | 0.069 | +0.062 |
| 25 | 4 | 0.532 | 0.073 | +0.067 |
| 50 | 4 | 0.508 | 0.019 | -0.019 |
| 100 | 4 | 0.541 | 0.095 | -0.013 |
| 200 | 4 | 0.539 | 0.082 | +0.008 |


_Metric = MCC / Accuracy on the FIXED held-out families. F1 is omitted: it is degenerate here (peaks at epoch 0, i.e. before learning). Near-chance across N (Acc ~0.51-0.54, MCC ~0.02-0.10), NO clean monotonic rise; last-epoch MCC ~0 or negative._

In [3]:
# DEFINITIVE test: famfull = ALL 346 families, 5 disjoint held-out folds (mmpartnet_out/famfull/)
import csv as _csv, statistics as _st
FF = OUT / 'famfull'
bestM = []; lastM = []; bestA = []
tab = '| fold | best MCC (epoch) | last MCC | best Acc |\n|---|---|---|---|\n'
for k in range(5):
    f = FF / str(k) / 'val_metrics.csv'
    if not f.exists(): continue
    rows = list(_csv.DictReader(open(f)))
    m = [float(r['MCC']) for r in rows]; a = [float(r['Accuracy']) for r in rows]
    bi = m.index(max(m)); tab += f'| {k} | {max(m):.3f} (ep{bi}) | {m[-1]:+.3f} | {max(a):.3f} |\n'
    bestM.append(max(m)); lastM.append(m[-1]); bestA.append(max(a))
display(Markdown(tab))
if bestM:
    display(Markdown(f'**FAMFULL (346 families, 5-fold): best-epoch MCC = {_st.mean(bestM):.3f} +/- {_st.pstdev(bestM):.3f} (Acc {_st.mean(bestA):.3f}); last-epoch MCC = {_st.mean(lastM):.3f} +/- {_st.pstdev(lastM):.3f}.** vs famscale N<=200 near-chance (MCC 0.02-0.10) -> diversity IS the lever.'))

| fold | best MCC (epoch) | last MCC | best Acc |
|---|---|---|---|
| 0 | 0.134 (ep3) | +0.134 | 0.555 |
| 1 | 0.589 (ep0) | +0.385 | 0.794 |
| 2 | 0.481 (ep2) | +0.479 | 0.737 |
| 3 | 0.376 (ep0) | +0.201 | 0.686 |
| 4 | 0.256 (ep3) | +0.256 | 0.605 |


**FAMFULL (346 families, 5-fold): best-epoch MCC = 0.367 +/- 0.161 (Acc 0.676); last-epoch MCC = 0.291 +/- 0.125.** vs famscale N<=200 near-chance (MCC 0.02-0.10) -> diversity IS the lever.

## Conclusion -- the null is OVERTURNED at maximal diversity
This curve is the **CORAL architecture** (ESM-2-150M + DNABERT2 + bidirectional cross-attention + LoRA) -- the diversity-enabled RNA-protein INTERACTION model, no PARNET body (no all-223 leakage here).

The famscale curve (N=10..200) was near-chance, which read as a null. But the DEFINITIVE test -- training on **ALL 346 families** (the full CORAL+Moyon pool), **5 disjoint held-out folds** -- generalizes to held-out FAMILIES: **best-epoch MCC 0.37 +/- 0.16 (Acc 0.68); last-epoch MCC 0.29 +/- 0.12**, far above the N<=200 near-chance. So the flat low-N curve was **UNDER-COVERAGE of RBP-space, not a ceiling**: **independent-family diversity IS the lever** -- exactly the interpolation-in-RBP-space prediction (sparse wells cannot interpolate; dense coverage can).

Honest reading:
1. **Metric** -- best-epoch peeks at the test; last-epoch (0.29) is the conservative number and still wins decisively. Folds 1/3 peak at epoch 0 then overfit training families -> a proper early-stop on a TRAIN-family val would land ~0.3-0.4.
2. **Mechanism (to verify, not hand-wave)** -- holding out 60 of 406 families leaves near-relatives in training, so this is likely dense-NEIGHBOUR interpolation (which IS the hypothesis) rather than transfer to isolated families. NEXT: near-vs-far distance-stratified check (does held-family MCC decay with sequence distance to the nearest training family?).
3. **Reconciles C1** -- CORAL's component-wise ~0.57 (paralog-permissive) vs family-held-out 0.29-0.37 (paralog-excluded, harder) now sit on ONE axis: performance scales with how densely RBP-space is covered near the held-out point.

### !! CONFOUND (see S2) -- this is NOT yet clean protein-family transfer
The held **RNAs are ~100% shared** with training (only proteins are held out), and an **RNA-only bindability baseline (protein ignored) reaches MCC ~0.23** vs CORAL's 0.37 -- so **most of this signal, and most of the famfull>famscale gain, is RNA-coverage memorization**, not protein-family interpolation (more families -> more training RNAs -> held RNAs better covered). The genuine protein-family residual is ~0.13 MCC (best-epoch), highly fold-variable. **Treat 'diversity is the lever' as CONFOUNDED pending the clean RNA-disjoint + family-disjoint split + protein-shuffle control.** See `notebooks/scaling/S3_rna_overlap`.

**Provenance:** CORAL fork trained on a CUDA GPU; per-fold results staged in `mmpartnet_out/famfull/` (RNAs truncated to 1000nt, batch 8, 4 epochs/fold, 5 disjoint 60-family held-out sets). famscale curve: `mmpartnet_out/famscale/`.